In [7]:
from IPython.display import IFrame
IFrame('empiricaldes.html', width='100%', height=600)

In [10]:
IFrame('results.html', width='100%', height=600)

In [35]:
import pandas as pd
import numpy as np

master = pd.read_csv("/Users/aliabonanno/public_policy_project/master_panel_monthly_fc.csv",
                     dtype={"FIPS": str})

#p10 = master["pct_below_150pct_poverty"].quantile(0.10)
#p90 = master["pct_below_150pct_poverty"].quantile(0.90)
#pov_deviation = p90 - p10

p25 = master["pct_below_150pct_poverty"].quantile(0.25)
p75 = master["pct_below_150pct_poverty"].quantile(0.75)
pov_deviation = p75 - p25
print(f"P25: {p25:.4f}")
print(f"P75: {p75:.4f}")
print(f"IQR deviation: {pov_deviation:.4f}")
mean_price_per_mwh = np.exp(master["log_implied_price"].mean())  # drop * 1000


summer_coop = master[
    (master["month"].isin([6, 7, 8])) &
    (master["dominant_ownership_type"] == "Cooperative")
].copy()

mean_b8 = summer_coop["bin8_25to30"].mean()
mean_b9 = summer_coop["bin9_over30"].mean()
mean_sales_per_customer = summer_coop["res_sales_per_customer"].mean()  # already in MWh

print(f"Mean implied price: ${mean_price_per_mwh:.2f}/MWh")
print(f"Mean days bin8: {mean_b8:.3f}")
print(f"Mean days bin9: {mean_b9:.3f}")
print(f"Mean sales per customer (geometric mean, MWh/month): {mean_sales_per_customer:.4f}")

coef_b8_coop = -0.025
coef_b9_coop = -0.038
coef_b8_base = -0.008
coef_b9_base = -0.014

log_shortfall_coop = (coef_b8_coop * mean_b8 + coef_b9_coop * mean_b9) * pov_deviation
log_shortfall_full = ((coef_b8_base + coef_b8_coop) * mean_b8 + (coef_b9_base + coef_b9_coop) * mean_b9) * pov_deviation

mwh_shortfall_coop = abs(log_shortfall_coop) * mean_sales_per_customer
mwh_shortfall_full = abs(log_shortfall_full) * mean_sales_per_customer

dollar_shortfall_coop = mwh_shortfall_coop * mean_price_per_mwh
dollar_shortfall_full = mwh_shortfall_full * mean_price_per_mwh


kwh_coop_summer = mwh_shortfall_coop * 3 * 1000
kwh_full_summer = mwh_shortfall_full * 3 * 1000
ac_hours_coop = kwh_coop_summer / 3.5
ac_hours_full = kwh_full_summer / 3.5

print(f"\nPer-household per summer:")
print(f"  Coop only — kWh: {kwh_coop_summer:.1f}, AC hrs: {ac_hours_coop:.1f}")
print(f"  Full gap  — kWh: {kwh_full_summer:.1f}, AC hrs: {ac_hours_full:.1f}")

coop_2023 = master[
    (master["month"].isin([6, 7, 8])) &
    (master["dominant_ownership_type"] == "Cooperative") &
    (master["year"] == 2023)
]
total_customers_2023 = coop_2023["res_customers_total"].sum()
agg_coop = dollar_shortfall_coop * total_customers_2023 * 3
agg_full = dollar_shortfall_full * total_customers_2023 * 3

print(f"\nAggregate (2023):")
print(f"  Total coop customers: {total_customers_2023:,.0f}")
print(f"  Coop amplification: ${agg_coop/1e6:.1f}M")
print(f"  Full gap:           ${agg_full/1e6:.1f}M")

P25: 0.1883
P75: 0.3031
IQR deviation: 0.1148
Mean implied price: $126.76/MWh
Mean days bin8: 11.910
Mean days bin9: 1.050
Mean sales per customer (geometric mean, MWh/month): 1.1429

Per-household per summer:
  Coop only — kWh: 132.9, AC hrs: 38.0
  Full gap  — kWh: 176.2, AC hrs: 50.3

Aggregate (2023):
  Total coop customers: 29,508,342
  Coop amplification: $497.2M
  Full gap:           $659.1M


In [17]:
print(master.columns.tolist())

['FIPS', 'year', 'month', 'res_revenue_000_total', 'res_sales_mwh_total', 'res_customers_total', 'n_utilities', 'implied_price_per_mwh', 'res_sales_per_customer', 'log_res_sales_per_customer', 'log_implied_price', 'dominant_ownership_type', 'bin1_under_neg5', 'bin2_neg5to0', 'bin3_0to5', 'bin4_5to10', 'bin5_10to15', 'bin6_15to20', 'bin7_20to25', 'bin8_25to30', 'bin9_over30', 'prcp_monthly_mm', 'prcp_monthly_mm_sq', 'pct_below_150pct_poverty', 'total_pop', 'median_hh_income', 'pct_white', 'pct_black', 'pct_hispanic', 'pct_bachelors_plus', 'unemployment_rate', 'pct_electric_heat', 'rucc_score', 'rural', 'fixed_charge_county', 'poverty_demeaned', 'fc_demeaned', 'covid', 'bin1_under_neg5_x_pov', 'bin2_neg5to0_x_pov', 'bin3_0to5_x_pov', 'bin4_5to10_x_pov', 'bin5_10to15_x_pov', 'bin7_20to25_x_pov', 'bin8_25to30_x_pov', 'bin9_over30_x_pov', 'bin7_20to25_x_pov_x_fc', 'bin8_25to30_x_pov_x_fc', 'bin9_over30_x_pov_x_fc', 'is_coop', 'is_municipal', 'bin7_20to25_x_pov_x_coop', 'bin7_20to25_x_pov_x_

In [29]:
print(master[master["dominant_ownership_type"] == "Cooperative"]["res_sales_per_customer"].mean())
print(summer_coop["res_sales_per_customer"].mean())
print(master["res_sales_per_customer"].mean())

1.0827940549492412
1.1428623154905553
0.9441965731984449


In [31]:
print(summer_coop["bin8_25to30"].describe())
print(summer_coop["bin9_over30"].describe())

count    4503.000000
mean       11.910282
std         8.417601
min         0.000000
25%         5.000000
50%        12.000000
75%        18.000000
max        31.000000
Name: bin8_25to30, dtype: float64
count    4503.000000
mean        1.049967
std         3.972969
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        31.000000
Name: bin9_over30, dtype: float64


In [37]:
print(master["res_sales_per_customer"].mean())  # overall
print(master[master["dominant_ownership_type"] == "Cooperative"]["res_sales_per_customer"].mean())  # coop all months
print(summer_coop["res_sales_per_customer"].mean())  # coop summer only

0.9441965731984449
1.0827940549492412
1.1428623154905553


In [39]:
print(summer_coop["res_customers_total"].describe())
print(coop_2023["res_customers_total"].sum())
print(coop_2023["res_customers_total"].mean())

count      4836.000000
mean      55518.152192
std       57189.507181
min          55.000000
25%       28776.250000
50%       35631.500000
75%       68822.500000
max      376975.000000
Name: res_customers_total, dtype: float64
29508342.0
59253.698795180724


In [41]:
print(dollar_shortfall_coop * 3)  # per household per summer
print((dollar_shortfall_coop * total_customers_2023 * 3) / (total_customers_2023 / 3))

16.84912101530356
50.54736304591067


In [43]:
# Get unique customer count — divide sum by 3 months
unique_customers_2023 = total_customers_2023 / 3

agg_coop = dollar_shortfall_coop * 3 * unique_customers_2023  # per month × 3 months × customers
agg_full = dollar_shortfall_full * 3 * unique_customers_2023

print(f"Unique coop customers: {unique_customers_2023:,.0f}")
print(f"Coop amplification: ${agg_coop/1e6:.1f}M")
print(f"Full gap: ${agg_full/1e6:.1f}M")

Unique coop customers: 9,836,114
Coop amplification: $165.7M
Full gap: $219.7M
